In [ ]:
import jax
import os

jax.config.update('jax_platform_name', 'cpu')

# # Avoid OOM errors
# os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"]=".33"
# # os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"]="false"
print(f"JAX backend: {jax.default_backend()}")

In [ ]:
import sys
sys.path.append("../")

# %load_ext autoreload
# %autoreload 2

import pulse_simulator as ps

In [ ]:
from datetime import datetime
import functools
import itertools
import logging
import numpy as np
import pickle as pkl
import qiskit as qk
from qiskit import pulse
import qiskit_aer
import qiskit.providers.fake_provider as qk_fp
import qiskit_dynamics as qk_d
import qutip as qt
import re 
from scipy.integrate import simpson
from scipy.linalg import sqrtm, expm
from tqdm import tqdm
import warnings

import matplotlib.pyplot as plt

In [ ]:
Units = 1e9

# Undo units
GHz = 1 / Units
MHz = 1e3 / Units
μs = Units / 1e3
ns = Units

backend = qk_fp.FakeManila()
dt = backend.configuration().dt * ns
desired_1qb_duration = 225  # dt
desired_2qb_duration = 225 * 4  # dt

# Gate time in SI units (granularity restriction)
duration_1qb = ps.qiskit_closest_multiple_of_16(desired_1qb_duration * dt / ns, backend)
duration_2qb = ps.qiskit_closest_multiple_of_16(desired_2qb_duration * dt / ns, backend)

# Load pulses

In [ ]:
# Calibrate the default X gates
default_gates = {}
for k in ["X1", "SX1", "I1", "X2", "SX2", "I2"]:
    if "SX" in k:
        default_gates[k] = ps.gaussian_envelope(duration_1qb, dt=dt, angle=np.pi / 4)
    elif "X" in k:
        default_gates[k] = ps.gaussian_envelope(duration_1qb, dt=dt, angle=np.pi / 2)
    elif "I" in k:
        default_gates[k] = ps.zero_envelope(duration_1qb)
    else:
        raise ValueError(f"Unknown gate {k}")
default_gates["I2"] = ps.zero_envelope(duration_1qb)


# Build circuit

In [ ]:
registers = [0]

my_moments = [{0: "X1"}]

my_schedule = ps.build_pulse_schedule(
    my_moments,
    default_gates,
    _,
    registers,
    backend,
    dt=dt,
    name="T1-simulation",
)

my_schedule.draw()

# Build model

In [ ]:
t1 = 250 / dt / μs * ns
damp_label = "D"

In [ ]:
t1 * dt

In [ ]:
1 / np.sqrt(t1) * ps.from_label(damp_label, reverse=True)

In [ ]:
# Set registers above with circuit
# The rest is automatic
# ===================== 

# NOTE: If the Rabi rates are True, you have to calibrate!
cvars = ps.backend_simulation_vars(backend, rabi=False, units=Units)

# Change T1 and T2 to match Eagle processors like Sherbrooke
for k, v in cvars.items():
    if "t1" in k:
        # cvars[k] = 250 / μs * ns
        cvars[k] = 250 / dt / μs * ns
    if "t2" in k:
        cvars[k] = None # 150 / μs * ns

# Carrier frequencies of each control line
# NOTE: We won't use carriers in this example
carriers = ps.backend_carriers(backend, cvars)

# Partially compile model functions with parameters
# =====
H_rx = functools.partial(
    ps.rx_model,
    registers=registers,
    backend=backend,
    variables=cvars,
    rotating_frame=True,
)

H_rz = functools.partial(
    ps.rz_model,
    registers=registers,
    backend=backend,
)

H_cr = functools.partial(
    ps.cross_resonance_model,
    registers=registers,
    backend=backend,
    variables=cvars,
    model_name="Toy",
)

A_decay = functools.partial(ps.qubit_decay_model, registers=registers, variables=cvars)

edges = ps.backend_edges(backend) #[1:-1]
print(f"Edges: {[e for e in edges if e[0] in registers and e[1] in registers]}")
H_xtalk = ps.crosstalk_model(registers, edges, cvars)

# Create a system model
# =====
H_drift = 0.0
Hs_control = []
Hs_channels = []
As_static = []
for qubit in registers:
    # Single qubit terms
    Hj_drift, Hjs_control, Hjs_channel = H_rx(qubit)
    Hj_phase, Hj_phase_ch = H_rz(qubit)
    Ajs_static = A_decay(qubit)

    H_drift += Hj_drift
    Hs_control += Hjs_control + [Hj_phase]
    Hs_channels += Hjs_channel + [Hj_phase_ch]
    As_static += Ajs_static

    # Two qubit with ring connectivity
    if qubit < len(registers) - 1:
        control, target = qubit, qubit + 1
        
        # Get both directions
        for pair in [(control, target), (target, control)]:
            Hj_drift, Hjs_control, Hjs_channel = H_cr(pair)
            H_drift += Hj_drift
            Hs_control += Hjs_control
            Hs_channels += Hjs_channel

# Some models may repeat control channels
# NOTE: This actually really matters! Repeat channels mess things up.
Hs_channels, indices = np.unique(Hs_channels, return_index=True)
Hs_control = [Hs_control[i] for i in indices]

# For RWA / Rotating frame
# NOTE: We won't use carriers in this example
# =====
registered_carriers = {c: 0 for c in Hs_channels}
cutoff_freq = None

# registered_carriers = {c: carriers[c] for c in Hs_channels}
# cutoff_freq = 2 * np.mean(list(registered_carriers.values()))

In [ ]:
As_static

In [ ]:
# measure_times = np.linspace(0, my_schedule.duration * dt, 10)

solver = qk_d.Solver(
    static_hamiltonian=None,
    hamiltonian_operators=Hs_control,
    hamiltonian_channels=Hs_channels,
    # static_dissipators=None,
    static_dissipators=As_static,
    rotating_frame=None,
    rwa_cutoff_freq=None,
    channel_carrier_freqs=registered_carriers,
    dt=dt
)

In [ ]:
y0 = ps.qiskit_identity_operator(len(registers))
solver.model.evaluation_mode = "dense"
res = solver.solve(
    t_span=[0.0, my_schedule.duration * dt],
    t_eval=[my_schedule.duration * dt],
    y0=y0,
    signals=my_schedule,
    atol=1e-9,
    rtol=1e-9,
    method="jax_odeint",
)

In [ ]:
res.y

In [ ]:
target = qk.quantum_info.Operator([[0, 1], [1, 0]])
[ps.quantum_info.process_fidelity(y, target) for y in res.y]

In [ ]:
tlist = np.arange(0, my_schedule.duration * dt, dt)
H_qutip =  [
    H_drift,
    [ps.get_samples(default_gates["X1"]),Hs_control[0]]
]

res_qt = qt.mesolve(
    H=H_qutip,
    rho0=qt.Qobj([[1, 0], [0, 0]]),
    tlist=tlist,
    # c_ops=[qt.Qobj(A.data) for A in As_static]
)

In [ ]:
# plt.plot(np.arange(0, my_schedule.duration * dt, dt), ps.get_samples(default_gates["X1"]))